# Contribution A p=0.5 Coherence Retraining

This notebook runs the real one-round M-OWODB Task-1 to Task-2 retraining experiment for `full` acquisition with `coherence_power = 0.5` using the actual DAOWOD package and the actual PROB bridge.

It uses the same Task-1 checkpoint, controlled long-tail construction, reference IDs, initially labelled IDs, budget, seed, acquisition settings, training settings, PROB configuration, and evaluation split as the previous Contribution A retraining experiments, but runs only the intermediate coherence-power operating point.

This focused retraining run is not a final benchmark result.

In [ ]:
# User-editable configuration.

SEED = 0
COHERENCE_POWER = 0.5
BUDGET = 10
SOURCE_TASK2_IMAGES = 300
REFERENCE_IMAGES = 30
EVAL_UNKNOWN_IMAGES = 100
EVAL_KNOWN_IMAGES = 100
IMBALANCE_RATIO = 20.0
TRAIN_EPOCHS = 1
BATCH_SIZE = 1
NUM_WORKERS = 2

ALPHA = 0.3
BETA = 0.2
GAMMA = 0.5
RARITY_POWER = 1.0
TOP_K = 3

STRATEGIES = ("full",)

ALLOW_OVERWRITE_DRIVE_RESULTS = False


# Repositories

DAOWOD_REPOSITORY_URL = (
    "https://github.com/gubiczam/"
    "distribution-aware-owod.git"
)

DAOWOD_COMMIT = "3f2763b2e54dd44b1cde27df5e2d8d87e54bf9e6"

PROB_REPOSITORY_URL = (
    "https://github.com/gubiczam/PROB.git"
)

PROB_BRANCH = "feat/daowod-bridge"


# Google Drive paths

DRIVE_ROOT = "/content/drive/MyDrive/DAOWOD"

DRIVE_ARCHIVE = (
    f"{DRIVE_ROOT}/assets/OWOD_full.tar.zst"
)

DRIVE_TASK1_CHECKPOINT = (
    f"{DRIVE_ROOT}/checkpoints/MOWODB/t1.pth"
)

EXPERIMENT_NAME = f"coherence_retraining_p05_seed{SEED}"

DRIVE_RESULT_DIR = (
    f"{DRIVE_ROOT}/results/{EXPERIMENT_NAME}"
)


# Local Colab paths

CONTENT_ROOT = "/content"

DAOWOD_PATH = (
    f"{CONTENT_ROOT}/distribution-aware-owod"
)

PROB_PATH = (
    f"{CONTENT_ROOT}/PROB"
)

DATA_ROOT = (
    f"{CONTENT_ROOT}/data/OWOD"
)

LOCAL_RESULT_ROOT = (
    f"{CONTENT_ROOT}/{EXPERIMENT_NAME}"
)

LOCAL_CHECKPOINT_DIR = (
    f"{CONTENT_ROOT}/daowod_checkpoints"
)

TASK1_CHECKPOINT_FOR_PROB = (
    f"{LOCAL_CHECKPOINT_DIR}/t1_epoch40.pth"
)


# Dataset and evaluation configuration

DATASET = "TOWOD"

TASK2_SOURCE_SPLIT = "owod_t2_train"

TASK1_REFERENCE_SPLIT = "owod_t1_train"

OFFICIAL_EVAL_SPLIT = "owod_all_task_test"

PILOT_EVAL_SPLIT = (
    f"daowod_pilot_balanced_seed{SEED}_test"
)

PREVIOUS_CLASSES = 20
CURRENT_CLASSES = 20
NUM_CLASSES = 81

MAX_PROPOSALS_PER_IMAGE = 20
MINIMUM_UNKNOWN_SCORE = 0.0

DEVICE = "cuda"


# Pipeline status tracking

STATUS_ORDER = (
    "GPU",
    "Drive assets",
    "system packages",
    "repositories",
    "DAOWOD install",
    "DAOWOD validation",
    "PROB bridge",
    "attention backend",
    "protocol construction",
    "evaluation split",
    "selective extraction",
    "full round",
    "Drive persistence",
)

STATUS = dict.fromkeys(
    STATUS_ORDER,
    "PENDING",
)

In [ ]:
import json
import platform
import random
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd


def run_cmd(command, *, cwd=None, timeout=600, check=True):
    command = [str(part) for part in command]
    print("$", " ".join(command))
    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=timeout,
        check=False,
    )
    if result.stdout:
        print(result.stdout[-8000:])
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, command, output=result.stdout)
    return result


def mark(stage):
    STATUS[stage] = "OK"
    print(f"{stage}: OK")


def read_ids(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing image set: {path}")
    return [line.split()[0] for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def write_ids(path, image_ids):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(image_ids) + ("\n" if image_ids else ""), encoding="utf-8")


def disk_free_gb(path="/content"):
    return round(shutil.disk_usage(path).free / 1024**3, 2)


import torch
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required. Select a Colab T4 GPU runtime.")

drive.mount("/content/drive", force_remount=False)
archive = Path(DRIVE_ARCHIVE)
task1_checkpoint = Path(DRIVE_TASK1_CHECKPOINT)
missing_drive_assets = [str(path) for path in (archive, task1_checkpoint) if not path.exists()]
if missing_drive_assets:
    raise FileNotFoundError("Missing Drive asset(s): " + ", ".join(missing_drive_assets))

runtime_rows = {
    "Python": sys.version.split()[0],
    "Platform": platform.platform(),
    "PyTorch": torch.__version__,
    "CUDA": torch.version.cuda,
    "GPU": torch.cuda.get_device_name(0),
    "Archive": str(archive),
    "Task-1 checkpoint": str(task1_checkpoint),
    "Free /content GB": disk_free_gb("/content"),
}
display(pd.DataFrame(runtime_rows.items(), columns=["item", "value"]))
mark("GPU")
mark("Drive assets")


In [ ]:
run_cmd(["apt-get", "update", "-qq"], timeout=600)
run_cmd(["apt-get", "install", "-y", "-qq", "zstd", "ninja-build", "build-essential"], timeout=900)
zstd_version = run_cmd(["zstd", "--version"], timeout=60).stdout.strip()
ninja_version = run_cmd(["ninja", "--version"], timeout=60).stdout.strip()
display(pd.DataFrame({"package": ["zstd", "ninja"], "version": [zstd_version, ninja_version]}))
mark("system packages")


In [ ]:
for checkout in (Path(DAOWOD_PATH), Path(PROB_PATH), Path(LOCAL_RESULT_ROOT), Path(LOCAL_CHECKPOINT_DIR)):
    if checkout.exists():
        shutil.rmtree(checkout)

run_cmd(["git", "clone", DAOWOD_REPOSITORY_URL, DAOWOD_PATH], cwd=CONTENT_ROOT, timeout=300)
run_cmd(["git", "checkout", DAOWOD_COMMIT], cwd=DAOWOD_PATH, timeout=120)
run_cmd(["git", "clone", "--branch", PROB_BRANCH, "--single-branch", PROB_REPOSITORY_URL, PROB_PATH], cwd=CONTENT_ROOT, timeout=300)
repo_commits = {
    "DAOWOD": run_cmd(["git", "rev-parse", "HEAD"], cwd=DAOWOD_PATH).stdout.strip(),
    "PROB": run_cmd(["git", "rev-parse", "HEAD"], cwd=PROB_PATH).stdout.strip(),
}
if repo_commits["DAOWOD"] != DAOWOD_COMMIT:
    raise RuntimeError(
        "DAOWOD checkout mismatch: "
        f"expected {DAOWOD_COMMIT}, got {repo_commits['DAOWOD']}"
    )
display(pd.DataFrame(repo_commits.items(), columns=["repository", "commit"]))
mark("repositories")

In [ ]:
import importlib

run_cmd([sys.executable, "-m", "pip", "install", "--editable", f"{DAOWOD_PATH}[dev]"], timeout=900)
daowod_src = str(Path(DAOWOD_PATH) / "src")
if daowod_src not in sys.path:
    sys.path.insert(0, daowod_src)
importlib.invalidate_caches()

import daowod
from daowod import ProbAdapter, load_config, run_active_round
from daowod.acquisition import AcquisitionWeights
from daowod.config import AcquisitionConfig
from daowod.dataset import build_long_tail_pool

imported_path = Path(daowod.__file__).resolve()
expected_root = (Path(DAOWOD_PATH) / "src" / "daowod").resolve()
print("daowod.__file__ =", imported_path)
if expected_root not in imported_path.parents and imported_path != expected_root / "__init__.py":
    raise RuntimeError(f"Imported daowod from unexpected location: {imported_path}")

base_config = load_config(Path(DAOWOD_PATH) / "configs" / "experiment.yaml")
base_acquisition = base_config.acquisition
acquisition_config = AcquisitionConfig(
    strategies=STRATEGIES,
    uncertainty_mode=base_acquisition.uncertainty_mode,
    pseudo_label_source=base_acquisition.pseudo_label_source,
    cluster_count=base_acquisition.cluster_count,
    neighbour_count=base_acquisition.neighbour_count,
    top_k=TOP_K,
    weights=AcquisitionWeights(
        uncertainty=ALPHA,
        novelty=BETA,
        rarity=GAMMA,
        coherence_power=COHERENCE_POWER,
        rarity_power=RARITY_POWER,
    ),
)
acquisition_rows = {
    "experiment_strategies": ", ".join(STRATEGIES),
    "uncertainty_mode": acquisition_config.uncertainty_mode,
    "pseudo_label_source": acquisition_config.pseudo_label_source,
    "cluster_count": acquisition_config.cluster_count,
    "neighbour_count": acquisition_config.neighbour_count,
    "top_k": acquisition_config.top_k,
    "alpha": acquisition_config.weights.uncertainty,
    "beta": acquisition_config.weights.novelty,
    "gamma": acquisition_config.weights.rarity,
    "coherence_power": acquisition_config.weights.coherence_power,
    "rarity_power": acquisition_config.weights.rarity_power,
}
display(pd.DataFrame(acquisition_rows.items(), columns=["acquisition_setting", "value"]))
mark("DAOWOD install")

In [ ]:
import importlib.util

prob_dependencies = {
    "wandb": "wandb",
    "einops": "einops",
    "pycocotools": "pycocotools",
    "skimage": "scikit-image",
    "joblib": "joblib",
    "tqdm": "tqdm",
    "ipdb": "ipdb",
}
missing_packages = [package for module, package in prob_dependencies.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    run_cmd([sys.executable, "-m", "pip", "install", *missing_packages], cwd=PROB_PATH, timeout=900)
else:
    print("PROB dependency imports already available.")

run_cmd([sys.executable, "-m", "ruff", "check", "."], cwd=DAOWOD_PATH, timeout=300)
run_cmd([sys.executable, "-m", "pytest", "-q"], cwd=DAOWOD_PATH, timeout=300)
run_cmd([sys.executable, "-m", "compileall", "-q", "src", "tests"], cwd=DAOWOD_PATH, timeout=300)
mark("DAOWOD validation")
run_cmd([sys.executable, "daowod_prob_bridge.py", "check"], cwd=PROB_PATH, timeout=120)
mark("PROB bridge")


In [ ]:
def require_text(path, needles, description):
    source = Path(path).read_text(encoding="utf-8")
    if not any(needle in source for needle in needles):
        raise RuntimeError(f"Missing {description}: {path}")
    compile(source, str(path), "exec")
    return source


def enable_attention_fallback():
    func_path = Path(PROB_PATH) / "models" / "ops" / "functions" / "ms_deform_attn_func.py"
    module_path = Path(PROB_PATH) / "models" / "ops" / "modules" / "ms_deform_attn.py"

    func_source = func_path.read_text(encoding="utf-8")
    original_import = "import MultiScaleDeformableAttention as MSDA"
    fallback_import = "try:\n    import MultiScaleDeformableAttention as MSDA\nexcept ModuleNotFoundError:\n    MSDA = None"
    if fallback_import in func_source:
        pass
    elif original_import in func_source:
        func_source = func_source.replace(original_import, fallback_import)
        func_path.write_text(func_source, encoding="utf-8")
    else:
        raise RuntimeError(f"Unknown attention function import form: {func_path}")

    module_source = module_path.read_text(encoding="utf-8")
    original_module_import = "from ..functions import MSDeformAttnFunction"
    fallback_module_import = "from ..functions.ms_deform_attn_func import MSDeformAttnFunction, ms_deform_attn_core_pytorch"
    if fallback_module_import in module_source:
        pass
    elif original_module_import in module_source:
        module_source = module_source.replace(original_module_import, fallback_module_import)
    else:
        raise RuntimeError(f"Unknown attention module import form: {module_path}")

    original_call = (
        "output = MSDeformAttnFunction.apply(\n"
        "            value, input_spatial_shapes, input_level_start_index, sampling_locations, attention_weights, self.im2col_step)"
    )
    fallback_call = (
        "output = ms_deform_attn_core_pytorch(\n"
        "            value, input_spatial_shapes, sampling_locations, attention_weights)"
    )
    if fallback_call in module_source:
        pass
    elif original_call in module_source:
        module_source = module_source.replace(original_call, fallback_call)
    else:
        raise RuntimeError(f"Unknown attention forward form: {module_path}")
    module_path.write_text(module_source, encoding="utf-8")
    print("Enabled pure-PyTorch deformable-attention fallback.")


def attention_forward_backward_check():
    code = "\n".join([
        "import torch",
        "from models.ops.modules.ms_deform_attn import MSDeformAttn",
        "assert torch.cuda.is_available()",
        "device = torch.device('cuda')",
        "module = MSDeformAttn(d_model=8, n_levels=1, n_heads=2, n_points=2).to(device)",
        "query = torch.randn(1, 3, 8, device=device, requires_grad=True)",
        "reference_points = torch.full((1, 3, 1, 2), 0.5, device=device)",
        "input_flatten = torch.randn(1, 4, 8, device=device, requires_grad=True)",
        "input_spatial_shapes = torch.tensor([[2, 2]], dtype=torch.long, device=device)",
        "input_level_start_index = torch.tensor([0], dtype=torch.long, device=device)",
        "output = module(query, reference_points, input_flatten, input_spatial_shapes, input_level_start_index)",
        "loss = output.square().mean()",
        "loss.backward()",
        "assert output.shape == (1, 3, 8)",
        "assert query.grad is not None and torch.isfinite(query.grad).all()",
        "assert input_flatten.grad is not None and torch.isfinite(input_flatten.grad).all()",
        "print('attention forward/backward OK', tuple(output.shape))",
    ])
    run_cmd([sys.executable, "-c", code], cwd=PROB_PATH, timeout=180)


require_text(Path(PROB_PATH) / "models" / "prob_deformable_detr.py", ["'pred_features': hs[-1]", '"pred_features": hs[-1]'], "decoder-feature export patch")
require_text(Path(PROB_PATH) / "main_open_world.py", ["test_stats = {}"], "main_open_world.py test_stats initialization fix")

compile_result = run_cmd(["bash", "make.sh"], cwd=Path(PROB_PATH) / "models" / "ops", timeout=900, check=False)
attention_backend = "CUDA extension"
if compile_result.returncode != 0:
    attention_backend = "pure PyTorch fallback"
    enable_attention_fallback()

attention_forward_backward_check()

backbone_path = Path(PROB_PATH) / "models" / "dino_resnet50_pretrain.pth"
if not backbone_path.exists():
    run_cmd([sys.executable, "-c", "import urllib.request; urllib.request.urlretrieve('https://dl.fbaipublicfiles.com/dino/dino_resnet50_pretrain/dino_resnet50_pretrain.pth', 'models/dino_resnet50_pretrain.pth')"], cwd=PROB_PATH, timeout=1800)
if not backbone_path.exists():
    raise FileNotFoundError(f"Missing DINO backbone: {backbone_path}")

Path(LOCAL_CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
checkpoint = torch.load(DRIVE_TASK1_CHECKPOINT, map_location="cpu", weights_only=False)
if isinstance(checkpoint, dict) and isinstance(checkpoint.get("epoch"), int):
    shutil.copy2(DRIVE_TASK1_CHECKPOINT, TASK1_CHECKPOINT_FOR_PROB)
    task1_epoch = checkpoint["epoch"]
elif isinstance(checkpoint, dict):
    wrapped = dict(checkpoint)
    wrapped["epoch"] = 40
    torch.save(wrapped, TASK1_CHECKPOINT_FOR_PROB)
    task1_epoch = 40
else:
    torch.save({"model": checkpoint, "epoch": 40}, TASK1_CHECKPOINT_FOR_PROB)
    task1_epoch = 40
verified = torch.load(TASK1_CHECKPOINT_FOR_PROB, map_location="cpu", weights_only=False)
if not isinstance(verified, dict) or not isinstance(verified.get("epoch"), int):
    raise RuntimeError("Task-1 checkpoint wrapper did not produce an integer epoch.")

display(pd.DataFrame({"item": ["attention backend", "Task-1 checkpoint", "Task-1 epoch", "DINO backbone"], "value": [attention_backend, TASK1_CHECKPOINT_FOR_PROB, task1_epoch, str(backbone_path)]}))
mark("attention backend")


In [ ]:
data_parent = Path(DATA_ROOT).parent
data_parent.mkdir(parents=True, exist_ok=True)
if Path(DATA_ROOT).exists():
    shutil.rmtree(DATA_ROOT)

run_cmd([
    "bash",
    "-lc",
    "zstd -dc \"$1\" | tar -xf - -C \"$2\" OWOD/ImageSets OWOD/Annotations",
    "bash",
    DRIVE_ARCHIVE,
    str(data_parent),
], timeout=1800)
required_paths = [
    Path(DATA_ROOT) / "ImageSets" / DATASET / f"{TASK2_SOURCE_SPLIT}.txt",
    Path(DATA_ROOT) / "ImageSets" / DATASET / f"{TASK1_REFERENCE_SPLIT}.txt",
    Path(DATA_ROOT) / "ImageSets" / DATASET / f"{OFFICIAL_EVAL_SPLIT}.txt",
    Path(DATA_ROOT) / "Annotations",
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError("Archive did not provide required protocol files: " + ", ".join(missing_paths))
annotation_count = sum(1 for _ in (Path(DATA_ROOT) / "Annotations").glob("*.xml"))
split_counts = {
    TASK2_SOURCE_SPLIT: len(read_ids(required_paths[0])),
    TASK1_REFERENCE_SPLIT: len(read_ids(required_paths[1])),
    OFFICIAL_EVAL_SPLIT: len(read_ids(required_paths[2])),
}
display(pd.DataFrame({"item": ["annotations", *split_counts], "count": [annotation_count, *split_counts.values()]}))


In [ ]:
import contextlib
import io
import xml.etree.ElementTree as ET

prob_src = str(Path(PROB_PATH).resolve())
if prob_src not in sys.path:
    sys.path.insert(0, prob_src)
with contextlib.redirect_stdout(io.StringIO()):
    from datasets.torchvision_datasets.open_world import (  # noqa: E402
        BASE_VOC_CLASS_NAMES,
        VOC_CLASS_NAMES_COCOFIED,
        VOC_COCO_CLASS_NAMES,
    )


def seeded_order_preserving_sample(image_ids, count, *, seed, salt):
    unique_ids = list(dict.fromkeys(str(image_id) for image_id in image_ids))
    if len(unique_ids) != len(image_ids):
        raise ValueError(f"{salt} source IDs contain duplicates.")
    if count > len(unique_ids):
        raise ValueError(f"Requested {count} images from only {len(unique_ids)} IDs for {salt}.")
    rng = random.Random(f"{seed}:{salt}")
    selected = set(rng.sample(unique_ids, count))
    return [image_id for image_id in unique_ids if image_id in selected]


image_set_root = Path(DATA_ROOT) / "ImageSets" / DATASET
official_task2_ids = read_ids(image_set_root / f"{TASK2_SOURCE_SPLIT}.txt")
task2_source_ids = seeded_order_preserving_sample(official_task2_ids, SOURCE_TASK2_IMAGES, seed=SEED, salt="task2-source")
pilot_source_split = image_set_root / "daowod_pilot_t2_source_train.txt"
write_ids(pilot_source_split, task2_source_ids)

task_class_names = list(VOC_COCO_CLASS_NAMES[DATASET][PREVIOUS_CLASSES : PREVIOUS_CLASSES + CURRENT_CLASSES])
long_tail_dir = Path(LOCAL_RESULT_ROOT) / "protocol" / "long_tail"
pool = build_long_tail_pool(
    annotation_dir=Path(DATA_ROOT) / "Annotations",
    source_split=pilot_source_split,
    task_class_names=task_class_names,
    output_dir=long_tail_dir,
    imbalance_ratio=IMBALANCE_RATIO,
    seed=SEED,
)
candidate_ids = [str(image_id) for image_id in pool["selected_image_ids"]]
official_t1_ids = read_ids(image_set_root / f"{TASK1_REFERENCE_SPLIT}.txt")
reference_source_ids = [image_id for image_id in official_t1_ids if image_id not in set(candidate_ids)]
reference_ids = seeded_order_preserving_sample(reference_source_ids, REFERENCE_IMAGES, seed=SEED, salt="reference")
labelled_ids = []

if len(candidate_ids) != len(set(candidate_ids)):
    raise RuntimeError("Candidate IDs are not unique.")
if len(reference_ids) != len(set(reference_ids)):
    raise RuntimeError("Reference IDs are not unique.")
if set(candidate_ids) & set(reference_ids):
    raise RuntimeError("Candidate and reference IDs overlap.")
if len(candidate_ids) < BUDGET:
    raise RuntimeError(f"Candidate pool has {len(candidate_ids)} images, below budget {BUDGET}.")
for artifact in (pool["pool_split_path"], pool["class_stats_path"], pool["manifest_path"]):
    if not Path(artifact).exists():
        raise FileNotFoundError(f"Missing protocol artifact: {artifact}")

training_protocol = {
    "seed": SEED,
    "source_split": str(pilot_source_split),
    "source_image_count": len(task2_source_ids),
    "candidate_pool_count": len(candidate_ids),
    "reference_count": len(reference_ids),
    "initial_labelled_count": len(labelled_ids),
    "imbalance_ratio": IMBALANCE_RATIO,
    "task_class_names": task_class_names,
    "candidate_ground_truth_visible_to_acquisition": False,
}
training_protocol_path = Path(LOCAL_RESULT_ROOT) / "protocol" / "training_protocol.json"
training_protocol_path.parent.mkdir(parents=True, exist_ok=True)
training_protocol_path.write_text(json.dumps(training_protocol, indent=2) + "\n", encoding="utf-8")
display(pd.DataFrame(training_protocol.items(), columns=["item", "value"]))
mark("protocol construction")


In [ ]:
def annotation_classes(image_id):
    path = Path(DATA_ROOT) / "Annotations" / f"{image_id}.xml"
    if not path.exists():
        raise FileNotFoundError(f"Missing annotation: {path}")
    root = ET.parse(path).getroot()
    classes = []
    for node in root.findall("./object/name"):
        if node.text:
            name = node.text.strip()
            if name in VOC_CLASS_NAMES_COCOFIED:
                name = BASE_VOC_CLASS_NAMES[VOC_CLASS_NAMES_COCOFIED.index(name)]
            classes.append(name)
    return classes


class_names = list(VOC_COCO_CLASS_NAMES[DATASET])
class_to_index = {name: index for index, name in enumerate(class_names)}
official_eval_ids_raw = read_ids(image_set_root / f"{OFFICIAL_EVAL_SPLIT}.txt")
official_eval_ids = list(dict.fromkeys(official_eval_ids_raw))
official_eval_duplicate_count = len(official_eval_ids_raw) - len(official_eval_ids)
unknown_candidates = []
known_only_candidates = []
unknown_object_count = 0
for image_id in official_eval_ids:
    indices = [class_to_index[name] for name in annotation_classes(image_id) if name in class_to_index]
    if not indices:
        continue
    has_unknown = any(40 <= index <= 79 for index in indices)
    known_only = all(0 <= index <= 39 for index in indices)
    if has_unknown:
        unknown_candidates.append(image_id)
        unknown_object_count += sum(40 <= index <= 79 for index in indices)
    elif known_only:
        known_only_candidates.append(image_id)

unknown_eval_ids = seeded_order_preserving_sample(unknown_candidates, EVAL_UNKNOWN_IMAGES, seed=SEED, salt="eval-unknown")
known_eval_ids = seeded_order_preserving_sample(known_only_candidates, EVAL_KNOWN_IMAGES, seed=SEED, salt="eval-known")
evaluation_selection = set(unknown_eval_ids) | set(known_eval_ids)
evaluation_ids = [image_id for image_id in official_eval_ids if image_id in evaluation_selection]
evaluation_split_path = image_set_root / f"{PILOT_EVAL_SPLIT}.txt"
write_ids(evaluation_split_path, evaluation_ids)

if "test" not in PILOT_EVAL_SPLIT:
    raise RuntimeError("Evaluation split name must contain 'test'.")
if len(evaluation_ids) != EVAL_UNKNOWN_IMAGES + EVAL_KNOWN_IMAGES:
    raise RuntimeError(f"Expected 200 evaluation IDs, got {len(evaluation_ids)}.")
if len(evaluation_ids) != len(set(evaluation_ids)):
    raise RuntimeError("Evaluation IDs contain duplicates.")
if unknown_object_count < 1:
    raise RuntimeError("Evaluation candidates contain no unknown ground-truth objects.")
for image_id in evaluation_ids:
    if not (Path(DATA_ROOT) / "Annotations" / f"{image_id}.xml").exists():
        raise FileNotFoundError(f"Missing evaluation XML: {image_id}")

evaluation_protocol = {
    "seed": SEED,
    "official_source_split": OFFICIAL_EVAL_SPLIT,
    "official_source_raw_count": len(official_eval_ids_raw),
    "official_source_unique_count": len(official_eval_ids),
    "official_source_duplicate_count": official_eval_duplicate_count,
    "pilot_split": PILOT_EVAL_SPLIT,
    "known_class_indices": "0..39",
    "unknown_class_indices": "40..79",
    "unknown_image_count": len(unknown_eval_ids),
    "known_only_image_count": len(known_eval_ids),
    "evaluation_count": len(evaluation_ids),
    "unknown_ground_truth_objects_in_source_candidates": unknown_object_count,
    "ground_truth_use": "evaluation_subset_construction_and_metrics_only",
}
evaluation_protocol_path = Path(LOCAL_RESULT_ROOT) / "protocol" / "evaluation_protocol.json"
evaluation_protocol_path.write_text(json.dumps(evaluation_protocol, indent=2) + "\n", encoding="utf-8")
display(pd.DataFrame(evaluation_protocol.items(), columns=["item", "value"]))
mark("evaluation split")

In [ ]:
required_image_ids = list(dict.fromkeys([*candidate_ids, *reference_ids, *evaluation_ids]))
member_list_path = Path(CONTENT_ROOT) / "daowod_required_jpegs.txt"
member_list_path.write_text(
    "\n".join(f"OWOD/JPEGImages/{image_id}.jpg" for image_id in required_image_ids) + "\n",
    encoding="utf-8",
)
run_cmd([
    "bash",
    "-lc",
    "zstd -dc \"$1\" | tar -xf - -C \"$2\" -T \"$3\"",
    "bash",
    DRIVE_ARCHIVE,
    str(Path(DATA_ROOT).parent),
    str(member_list_path),
], timeout=3600)
missing_required = []
for image_id in required_image_ids:
    for relative in (f"JPEGImages/{image_id}.jpg", f"Annotations/{image_id}.xml"):
        path = Path(DATA_ROOT) / relative
        if not path.exists():
            missing_required.append(str(path))
if missing_required:
    raise FileNotFoundError("Selective extraction missing required files: " + ", ".join(missing_required[:20]))

jpeg_size_gb = round(sum((Path(DATA_ROOT) / "JPEGImages" / f"{image_id}.jpg").stat().st_size for image_id in required_image_ids) / 1024**3, 3)
extraction_rows = {
    "candidate count": len(candidate_ids),
    "reference count": len(reference_ids),
    "evaluation count": len(evaluation_ids),
    "unique required JPEG count": len(required_image_ids),
    "extracted JPEG GB": jpeg_size_gb,
    "free /content GB": disk_free_gb("/content"),
}
display(pd.DataFrame(extraction_rows.items(), columns=["item", "value"]))
mark("selective extraction")


In [ ]:
python_command = sys.executable
common_prob_args = (
    f"--data-root {DATA_ROOT} --dataset {DATASET} "
    f"--prev-introduced-classes {PREVIOUS_CLASSES} --current-introduced-classes {CURRENT_CLASSES} "
    f"--num-classes {NUM_CLASSES} --batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS} "
    f"--device {DEVICE} --seed {SEED}"
)
adapter = ProbAdapter(
    repository_path=PROB_PATH,
    timeout_seconds=86400,
    train_command=(
        f"{python_command} daowod_prob_bridge.py train "
        "--labelled-ids {labelled_ids} --previous-checkpoint {previous_checkpoint} "
        "--output-checkpoint {checkpoint} --output-dir {output_dir} "
        f"--test-set {PILOT_EVAL_SPLIT} --epochs {TRAIN_EPOCHS} {common_prob_args}"
    ),
    predict_command=(
        f"{python_command} daowod_prob_bridge.py predict "
        "--image-ids {image_ids} --checkpoint {checkpoint} --output {proposals} "
        f"--max-proposals-per-image {MAX_PROPOSALS_PER_IMAGE} "
        f"--minimum-unknown-score {MINIMUM_UNKNOWN_SCORE} {common_prob_args}"
    ),
    evaluate_command=(
        f"{python_command} daowod_prob_bridge.py evaluate "
        "--checkpoint {checkpoint} --output {metrics} --output-dir {output_dir} "
        f"--test-set {PILOT_EVAL_SPLIT} {common_prob_args}"
    ),
)
configuration_rows = {
    "seed": SEED,
    "coherence power": COHERENCE_POWER,
    "budget": BUDGET,
    "candidate pool count": len(candidate_ids),
    "reference images": len(reference_ids),
    "initial labelled images": len(labelled_ids),
    "evaluation images": len(evaluation_ids),
    "Task-1 checkpoint": TASK1_CHECKPOINT_FOR_PROB,
    "strategies": ", ".join(STRATEGIES),
    "train epochs": TRAIN_EPOCHS,
    "batch size": BATCH_SIZE,
    "workers": NUM_WORKERS,
    "alpha": acquisition_config.weights.uncertainty,
    "beta": acquisition_config.weights.novelty,
    "gamma": acquisition_config.weights.rarity,
    "coherence power": acquisition_config.weights.coherence_power,
    "rarity power": acquisition_config.weights.rarity_power,
    "top k": acquisition_config.top_k,
}
display(pd.DataFrame(configuration_rows.items(), columns=["setting", "value"]))

if STRATEGIES != ("full",):
    raise RuntimeError("This notebook is dedicated to the full p=0.5 retraining strategy.")

strategy = "full"
strategy_output_dir = Path(LOCAL_RESULT_ROOT) / strategy
manifest_path = strategy_output_dir / "round_manifest.json"
if manifest_path.exists() and json.loads(manifest_path.read_text(encoding="utf-8")).get("completed") is True:
    raise RuntimeError(f"Completed round already exists and will not be overwritten: {strategy_output_dir}")
round_results = {
    strategy: run_active_round(
        adapter=adapter,
        checkpoint=TASK1_CHECKPOINT_FOR_PROB,
        candidate_ids=list(candidate_ids),
        reference_ids=list(reference_ids),
        labelled_ids=list(labelled_ids),
        output_dir=strategy_output_dir,
        strategy=strategy,
        budget=BUDGET,
        acquisition_config=acquisition_config,
        seed=SEED,
        round_index=0,
    )
}
mark("full round")

In [ ]:
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd


REQUIRED_METRICS = {
    "known_mAP",
    "U_Recall",
    "WI",
    "A_OSE",
    "unknown_AP50",
    "previous_known_AP50",
}


def json_default(value):
    """Convert NumPy and Path values into standard JSON-compatible values."""
    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, Path):
        return str(value)

    raise TypeError(
        f"Object of type {type(value).__name__} "
        "is not JSON serializable"
    )


def load_round(strategy):
    directory = Path(LOCAL_RESULT_ROOT) / strategy

    metrics_path = directory / "metrics.json"
    manifest_path = directory / "round_manifest.json"
    selected_path = directory / "selected_ids.txt"
    labelled_path = directory / "labelled_ids.txt"
    remaining_path = directory / "remaining_pool_ids.txt"

    required_files = [
        metrics_path,
        manifest_path,
        selected_path,
        labelled_path,
        remaining_path,
        directory / "checkpoint.pth",
        directory / "candidate_proposals.npz",
        directory / "reference_proposals.npz",
        directory / "proposal_scores.csv",
        directory / "image_scores.csv",
    ]

    missing_files = [
        str(path)
        for path in required_files
        if not path.exists()
    ]

    if missing_files:
        raise FileNotFoundError(
            f"Missing {strategy} round artifacts: "
            + ", ".join(missing_files)
        )

    metrics = json.loads(
        metrics_path.read_text(encoding="utf-8")
    )

    manifest = json.loads(
        manifest_path.read_text(encoding="utf-8")
    )

    selected = read_ids(selected_path)
    labelled = read_ids(labelled_path)
    remaining = read_ids(remaining_path)

    return (
        directory,
        metrics,
        manifest,
        selected,
        labelled,
        remaining,
    )


if STRATEGIES != ("full",):
    raise RuntimeError("This notebook validates only the full p=0.5 retraining round.")

strategy = "full"
directory, metrics, manifest, selected, labelled, remaining = load_round(strategy)

assert manifest["completed"] is True, "full round did not complete"
assert manifest["strategy"] == strategy, "manifest has wrong strategy"
assert manifest["input_checkpoint"] == TASK1_CHECKPOINT_FOR_PROB
assert manifest["candidate_count_before"] == len(candidate_ids)
assert manifest["budget"] == BUDGET
assert manifest["seed"] == SEED
assert len(selected) == BUDGET, f"full selected {len(selected)} images"
assert len(set(selected)) == BUDGET, "full selected duplicate images"
assert REQUIRED_METRICS <= set(metrics), "metrics missing required fields"
assert set(selected) <= set(candidate_ids), "full selected outside candidate pool"
assert set(selected).isdisjoint(remaining), "selected IDs still remain in pool"
assert labelled[-BUDGET:] == selected, "selected IDs were not revealed in order"

acquisition_parameters = manifest.get("acquisition_parameters")
if acquisition_parameters is not None:
    manifest_coherence_power = acquisition_parameters.get("coherence_power")
    weights = acquisition_parameters.get("weights", {})
    if manifest_coherence_power is None and isinstance(weights, dict):
        manifest_coherence_power = weights.get("coherence_power")
    assert manifest_coherence_power is not None, "manifest omits coherence_power"
    assert float(manifest_coherence_power) == COHERENCE_POWER

assert len(candidate_ids) == len(set(candidate_ids))
assert len(reference_ids) == len(set(reference_ids))
assert set(candidate_ids).isdisjoint(reference_ids)
assert len(remaining) == len(candidate_ids) - BUDGET

assert (
    len(evaluation_ids)
    == len(set(evaluation_ids))
    == EVAL_UNKNOWN_IMAGES + EVAL_KNOWN_IMAGES
)

with np.load(
    directory / "candidate_proposals.npz",
    allow_pickle=True,
) as proposals:
    proposal_ids = [
        str(value)
        for value in proposals["image_ids"].tolist()
    ]

assert set(proposal_ids) == set(candidate_ids), "proposal IDs do not match candidate IDs"
assert manifest["reference_proposals_sha256"], "reference proposal hash missing"

comparison = pd.DataFrame(
    [
        {
            "seed": int(SEED),
            "strategy": strategy,
            "coherence_power": float(COHERENCE_POWER),
            "selected_images": int(len(selected)),
            "known_mAP": float(metrics["known_mAP"]),
            "U_Recall": float(metrics["U_Recall"]),
            "WI": float(metrics["WI"]),
            "A_OSE": int(metrics["A_OSE"]),
            "unknown_AP50": float(metrics["unknown_AP50"]),
            "previous_known_AP50": float(metrics["previous_known_AP50"]),
            "completed": bool(manifest["completed"]),
        }
    ]
)

metric_columns = [
    "seed",
    "strategy",
    "coherence_power",
    "selected_images",
    "known_mAP",
    "U_Recall",
    "WI",
    "A_OSE",
    "unknown_AP50",
    "previous_known_AP50",
    "completed",
]

display(comparison[metric_columns])


# ------------------------------------------------------------------
# Save reproducible experiment summary
# ------------------------------------------------------------------

experiment_summary = {
    "seed": int(SEED),
    "coherence_power": float(COHERENCE_POWER),
    "budget": int(BUDGET),
    "source_task2_image_count": int(SOURCE_TASK2_IMAGES),
    "candidate_pool_count": int(len(candidate_ids)),
    "reference_count": int(len(reference_ids)),
    "evaluation_count": int(len(evaluation_ids)),
    "imbalance_ratio": float(IMBALANCE_RATIO),
    "strategy": strategy,
    "metric_summary": comparison.to_dict(orient="records"),
    "selected_ids": list(selected),
    "DAOWOD_commit": str(repo_commits["DAOWOD"]),
    "PROB_commit": str(repo_commits["PROB"]),
    "benchmark_result": False,
}

summary_path = (
    Path(LOCAL_RESULT_ROOT)
    / "experiment_summary.json"
)

summary_path.write_text(
    json.dumps(
        experiment_summary,
        indent=2,
        default=json_default,
    )
    + "\n",
    encoding="utf-8",
)

print(f"Experiment summary saved locally: {summary_path}")


# Verify that the written JSON is valid.
saved_summary = json.loads(
    summary_path.read_text(encoding="utf-8")
)

assert saved_summary["seed"] == int(SEED)
assert saved_summary["coherence_power"] == float(COHERENCE_POWER)
assert saved_summary["budget"] == int(BUDGET)
assert saved_summary["strategy"] == strategy
assert saved_summary["benchmark_result"] is False


# ------------------------------------------------------------------
# Copy completed experiment to Google Drive
# ------------------------------------------------------------------

destination = Path(DRIVE_RESULT_DIR)

if destination.exists():
    if not ALLOW_OVERWRITE_DRIVE_RESULTS:
        raise FileExistsError(
            "Drive result directory already exists: "
            f"{destination}. Rename or remove only this "
            "experiment directory, or set "
            "ALLOW_OVERWRITE_DRIVE_RESULTS=True."
        )

    shutil.rmtree(destination)


destination.parent.mkdir(
    parents=True,
    exist_ok=True,
)

shutil.copytree(
    LOCAL_RESULT_ROOT,
    destination,
)


# The extracted dataset must never be persisted with the experiment.
if (
    (destination / "data").exists()
    or (destination / "OWOD").exists()
):
    raise RuntimeError(
        "Dataset extraction unexpectedly appeared "
        "in the Drive result directory."
    )


copied_manifest_path = (
    destination
    / strategy
    / "round_manifest.json"
)

copied_manifest = json.loads(
    copied_manifest_path.read_text(
        encoding="utf-8"
    )
)

if copied_manifest.get("completed") is not True:
    raise RuntimeError("Copied incomplete full results to Drive.")

copied_summary_path = (
    destination
    / "experiment_summary.json"
)

if not copied_summary_path.exists():
    raise FileNotFoundError(
        f"Copied summary is missing: {copied_summary_path}"
    )

json.loads(
    copied_summary_path.read_text(encoding="utf-8")
)


display(
    pd.DataFrame(
        {
            "saved_to": [str(destination)],
            "summary": [str(copied_summary_path)],
        }
    )
)

mark("Drive persistence")

In [ ]:
final_status = pd.DataFrame([{"stage": stage, "status": STATUS[stage]} for stage in STATUS_ORDER])
display(final_status)
if not all(STATUS[stage] == "OK" for stage in STATUS_ORDER):
    raise RuntimeError("Contribution A p=0.5 retraining finished with one or more non-OK stages.")
print("Contribution A p=0.5 retraining completed successfully.")